# DSS for Agricultural Production and Farmer Management: Pagsanjan, Laguna
**Machine learning notebook (Colab)**

Modules (from Chapter I and III of the thesis):
1. **Harvest and income estimation**: land area is the main farmer input. Models compared: crop-average baseline, Elastic Net, Random Forest, Gaussian Process Regression (GPR). Metrics: MAE, RMSE, MAPE.
2. **Price forecast and selling advisory**: models compared: seasonal-naive baseline, ARIMA, Elastic Net, Random Forest, simple ensemble.
3. **Weather-based planting advisory**: agroclimatic matching (rule-based, FAO 1978 concept).
4. **Supabase**: precomputed tables the Flutter app reads directly (no Python server needed), plus export files.

> **IMPORTANT:** `DATA_SOURCE = "sample"` uses **synthetic data** just to test that the pipeline runs. The metrics from it mean nothing for your thesis. Set it to `"supabase"` (reads the MAO records from your Supabase tables) or `"csv"` (upload files) to get real results.

### Real data format (2 Supabase tables or CSVs + automatic weather)
| Table / file | Columns |
|---|---|
| Supabase `historical_records` (`crop, year, month, production_mt, price_per_kg`, plus **`area_harvested_ha`**) or `production.csv` | `year, month, barangay, crop, area_ha, production_kg` (month = harvest month, 1-12) |
| `prices.csv` (csv mode only) | `year, month, crop, price_per_kg` |
| weather | **Downloaded automatically from Open-Meteo** (free, no API key) for Pagsanjan: monthly `rainfall_mm` and `temp_c` |

## 1. Setup

In [ ]:
!pip -q install statsmodels scikit-learn joblib

import os, json, shutil, calendar, warnings
import numpy as np, pandas as pd, joblib
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

## 2. Connect to Supabase (get the MAO records, send the results)
In Colab click the 🔑 **Secrets** icon and add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY` (Supabase → Project Settings → API), and turn on *Notebook access*. The app's tables and the result tables must already exist (see `sql/app_schema_changes.sql`). The service key bypasses security rules, so keep it only in Colab Secrets.

In [ ]:
import subprocess, sys

def get_sb():
    try:
        from supabase import create_client
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "supabase"], check=True)
        from supabase import create_client
    from google.colab import userdata
    return create_client(userdata.get("SUPABASE_URL"), userdata.get("SUPABASE_SERVICE_KEY"))

def fetch_table(sb, table, page=1000):
    """GET: read a whole table (Supabase returns max 1000 rows per request, so page through it)."""
    rows, i = [], 0
    while True:
        r = sb.table(table).select("*").range(i, i + page - 1).execute()
        rows += r.data
        if len(r.data) < page:
            break
        i += page
    return pd.DataFrame(rows)

def send_rows(sb, table, rows, conflict=None, chunk=500):
    """SEND: insert rows, or upsert when a conflict key is given."""
    rows = json.loads(json.dumps(rows, default=lambda o: o.item()))   # numpy -> plain Python
    for i in range(0, len(rows), chunk):
        q = sb.table(table)
        (q.upsert(rows[i:i+chunk], on_conflict=conflict) if conflict else q.insert(rows[i:i+chunk])).execute()
    print("sent", len(rows), "rows to", table)

# ---- adapter for the app's own tables (historical_records + farms) ----
MONTHS = {m[:3].lower(): i for i, m in enumerate(calendar.month_abbr) if m}
def month_to_int(v):
    t = str(v).strip().lower()
    return int(t) if t.isdigit() else MONTHS.get(t[:3], np.nan)
def norm_crop(series):
    return series.astype(str).str.strip().str.title()

def load_from_app_tables(sb):
    """historical_records (municipality-wide, per crop and month) -> (production, prices) used by the models.
    Yield per hectare = production / area harvested. Area comes from historical_records.area_harvested_ha if it
    exists, otherwise it is APPROXIMATED by the total registered farm area of that crop (farms table)."""
    h = fetch_table(sb, "historical_records")
    assert len(h), "historical_records is empty."
    h["crop"] = norm_crop(h.crop)
    h["month"] = h.month.map(month_to_int)
    h["year"] = pd.to_numeric(h.year, errors="coerce")
    prices = h[["year", "month", "crop", "price_per_kg"]].copy()
    if "area_harvested_ha" in h.columns and h.area_harvested_ha.notna().any():
        area = pd.to_numeric(h.area_harvested_ha, errors="coerce")
        print("area source: historical_records.area_harvested_ha")
    else:
        farms = fetch_table(sb, "farms")
        assert len(farms), "No area_harvested_ha column and the farms table is empty."
        farms["crop_type"] = norm_crop(farms.crop_type)
        area = h.crop.map(farms.groupby("crop_type").land_area_hectares.sum())
        print("WARNING: no area_harvested_ha column -> using total registered farm area per crop as an APPROXIMATION.")
    production = pd.DataFrame({"year": h.year, "month": h.month, "barangay": "ALL", "crop": h.crop, "area_ha": area,
                               "production_kg": pd.to_numeric(h.production_mt, errors="coerce") * 1000})
    return production, prices

## 2b. Load data

In [ ]:
DATA_SOURCE = "sample"   # "sample" (synthetic test data) | "supabase" (your historical_records table) | "csv" (upload files)

if DATA_SOURCE == "sample":
    rng = np.random.default_rng(42)
    years = list(range(2015, 2025))
    barangays = [f"Barangay {i}" for i in range(1, 17)]       # placeholder names
    crops = {  # base yield kg/ha, base price PHP/kg
        "Rice":     (4000, 20), "Tomato": (12000, 40), "Eggplant": (10000, 45),
        "Banana":  (15000, 25), "Okra":   (6000, 35)}

    wrows = []
    for y in years:
        for m in range(1, 13):
            rain = 220 + 180*np.sin(2*np.pi*(m-4.5)/12) + rng.normal(0, 40)
            temp = 27.5 + 3*np.sin(2*np.pi*(m-2)/12) + rng.normal(0, 0.6)
            wrows.append((y, m, max(rain, 10), temp))
    weather = pd.DataFrame(wrows, columns=["year", "month", "rainfall_mm", "temp_c"])

    prows = []
    for c, (_, bp) in crops.items():
        phase = rng.uniform(0, 12)
        for y in years:
            for m in range(1, 13):
                p = bp*(1 + 0.18*np.sin(2*np.pi*(m+phase)/12) + 0.03*(y-2015)) + rng.normal(0, bp*0.04)
                prows.append((y, m, c, round(p, 2)))
    prices = pd.DataFrame(prows, columns=["year", "month", "crop", "price_per_kg"])

    w = weather.set_index(["year", "month"])
    rows = []
    for y in years:
        for c, (by, _) in crops.items():
            for b in barangays:
                for _ in range(2):
                    m = int(rng.integers(1, 13)); area = float(np.round(rng.lognormal(0, 0.5), 2))
                    r, t = w.loc[(y, m)]
                    yld = by*(1 + 0.0005*(r-220) - 0.03*abs(t-27.5)) * rng.normal(1, 0.08)
                    rows.append((y, m, b, c, area, round(yld*area, 1)))
    production = pd.DataFrame(rows, columns=["year", "month", "barangay", "crop", "area_ha", "production_kg"])
elif DATA_SOURCE == "supabase":
    sb = get_sb()
    production, prices = load_from_app_tables(sb)      # reads your historical_records (+ farms) tables
    weather = None          # weather comes from Open-Meteo, next section
else:
    from google.colab import files
    files.upload()   # upload production.csv and prices.csv
    production = pd.read_csv("production.csv")
    prices = pd.read_csv("prices.csv")
    weather = None

print(production.shape, prices.shape)
production.head()

## 2c. Weather from Open-Meteo
Daily historical weather for Pagsanjan is downloaded from the Open-Meteo archive API (free, no key needed) and summed or averaged into **monthly rainfall (mm)** and **mean temperature (°C)**. This needs internet, which Colab has. The same service also gives a 16-day forecast for the live advisory.

In [ ]:
import requests
USE_OPEN_METEO = True
LAT, LON = 14.2736, 121.4569          # Pagsanjan, Laguna (approx. town center)

def fetch_open_meteo_monthly(start_year, end_year, lat=LAT, lon=LON):
    last_ok = pd.Timestamp.today().normalize() - pd.Timedelta(days=7)      # archive lags a few days
    end = min(pd.Timestamp(f"{end_year}-12-31"), last_ok)
    r = requests.get("https://archive-api.open-meteo.com/v1/archive", timeout=120, params={
        "latitude": lat, "longitude": lon,
        "start_date": f"{start_year}-01-01", "end_date": end.strftime("%Y-%m-%d"),
        "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Asia/Manila"})
    r.raise_for_status()
    d = pd.DataFrame(r.json()["daily"]); d["time"] = pd.to_datetime(d["time"])
    d["year"], d["month"] = d.time.dt.year, d.time.dt.month
    m = (d.groupby(["year", "month"])
           .agg(rainfall_mm=("precipitation_sum", "sum"), temp_c=("temperature_2m_mean", "mean"), days=("time", "size"))
           .reset_index())
    m = m[m.days == m.apply(lambda r: calendar.monthrange(int(r.year), int(r.month))[1], axis=1)]   # drop incomplete months
    return m.drop(columns="days").round(1)

def fetch_open_meteo_forecast(days=16, lat=LAT, lon=LON):
    r = requests.get("https://api.open-meteo.com/v1/forecast", timeout=60, params={
        "latitude": lat, "longitude": lon, "forecast_days": days,
        "daily": "temperature_2m_mean,precipitation_sum", "timezone": "Asia/Manila"})
    r.raise_for_status()
    return pd.DataFrame(r.json()["daily"]).rename(columns={"time": "date", "temperature_2m_mean": "temp_c", "precipitation_sum": "rainfall_mm"})

if USE_OPEN_METEO or weather is None:
    weather = fetch_open_meteo_monthly(int(production.year.min()), int(production.year.max()))
    weather.to_csv("weather_open_meteo.csv", index=False)
print(weather.shape); display(weather.head())

# 16-day outlook (shown for reference; the app can use this for live advisories)
try:
    display(fetch_open_meteo_forecast().head(7))
except Exception as e:
    print("Forecast not available:", e)

## 3. Data cleaning and preparation (Stage 3 of the conceptual framework)

In [ ]:
need = {"production": (production, {"year","month","barangay","crop","area_ha","production_kg"}),
        "prices": (prices, {"year","month","crop","price_per_kg"}),
        "weather": (weather, {"year","month","rainfall_mm","temp_c"})}
for name, (d, cols) in need.items():
    missing = cols - set(d.columns)
    assert not missing, f"{name} is missing columns: {missing}"

for col in ["year", "month", "area_ha", "production_kg"]:
    production[col] = pd.to_numeric(production[col], errors="coerce")
for col in ["year", "month", "price_per_kg"]:
    prices[col] = pd.to_numeric(prices[col], errors="coerce")

production = production.dropna(subset=["year","month","crop","area_ha","production_kg"])
production = production[(production.area_ha > 0) & (production.production_kg > 0)].copy()
production["yield_kg_ha"] = production.production_kg / production.area_ha

# remove extreme outliers per crop (3 x IQR)
q1 = production.groupby("crop")["yield_kg_ha"].transform(lambda s: s.quantile(.25))
q3 = production.groupby("crop")["yield_kg_ha"].transform(lambda s: s.quantile(.75))
iqr = q3 - q1
production = production[(production.yield_kg_ha >= q1 - 3*iqr) & (production.yield_kg_ha <= q3 + 3*iqr)]

clim = weather.groupby("month")[["rainfall_mm", "temp_c"]].mean()   # monthly climatology of Pagsanjan

df = production.merge(weather, on=["year", "month"], how="left")
for c in ["rainfall_mm", "temp_c"]:
    df[c] = df[c].fillna(df["month"].map(clim[c]))
df["month_sin"] = np.sin(2*np.pi*df.month/12)
df["month_cos"] = np.cos(2*np.pi*df.month/12)
print(df.shape); df.head()

## 4. Module 1: Harvest estimation
The model predicts **yield per hectare** from crop, barangay, harvest month, and weather. **Harvest = yield per ha × land area**. The test set is the **last 2 years** (time-based split, which is more honest than a random split).

In [ ]:
CAT = ["crop", "barangay"]
NUM = ["month_sin", "month_cos", "rainfall_mm", "temp_c"]
FEATS = CAT + NUM

last_year = df.year.max()
train, test = df[df.year < last_year - 1], df[df.year >= last_year - 1]
print("train rows:", len(train), "| test rows:", len(test))

def prep():
    return ColumnTransformer([("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT),
                              ("num", StandardScaler(), NUM)])

def metrics(y, p):
    y, p = np.asarray(y, float), np.asarray(p, float)
    return {"MAE": mean_absolute_error(y, p),
            "RMSE": float(np.sqrt(mean_squared_error(y, p))),
            "MAPE_%": float(np.mean(np.abs((y - p) / y)) * 100)}

def make_harvest_model(name):
    est = {"ElasticNet": ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=20000),
           "RandomForest": RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1),
           "GaussianProcess": GaussianProcessRegressor(kernel=C(1.0)*RBF(length_scale=3.0) + WhiteKernel(0.1),
                                                       normalize_y=True, random_state=42)}[name]
    return Pipeline([("prep", prep()), ("model", est)])

results = {}
base_pred = test.crop.map(train.groupby("crop").yield_kg_ha.mean())       # baseline: average yield per crop
results["Baseline (crop mean)"] = metrics(test.yield_kg_ha, base_pred)
for name in ["ElasticNet", "RandomForest", "GaussianProcess"]:
    m = make_harvest_model(name).fit(train[FEATS], train.yield_kg_ha)
    results[name] = metrics(test.yield_kg_ha, m.predict(test[FEATS]))

harvest_results = pd.DataFrame(results).T.sort_values("RMSE")
print("Yield (kg/ha) error on the test set:"); display(harvest_results.round(2))

best_harvest = harvest_results.drop(index="Baseline (crop mean)").index[0]
print("Selected harvest model:", best_harvest)
harvest_model = make_harvest_model(best_harvest).fit(df[FEATS], df.yield_kg_ha)   # refit on all data
harvest_rmse = harvest_results.loc[best_harvest, "RMSE"]

## 5. Module 2: Price forecast and selling advisory
For each crop, the last 12 months are held out. Every model forecasts those 12 months, and the best one (lowest RMSE) is refit on the full history to forecast the **next 12 months**.

In [ ]:
price_df = prices.copy()
price_df["date"] = pd.to_datetime(dict(year=price_df.year, month=price_df.month, day=1))
series = {c: g.sort_values("date").set_index("date").price_per_kg.asfreq("MS").interpolate()
          for c, g in price_df.groupby("crop")}

def feats(hist, month):
    return [hist[-1], hist[-2], hist[-3], hist[-12], np.sin(2*np.pi*month/12), np.cos(2*np.pi*month/12)]

def make_price_ml(name):
    if name == "ElasticNet":
        return Pipeline([("sc", StandardScaler()), ("m", ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=20000))])
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=42, n_jobs=-1)

def fit_forecast(name, s, steps=12):
    idx = pd.date_range(s.index[-1] + pd.offsets.MonthBegin(1), periods=steps, freq="MS")
    vals = s.values.astype(float)
    if name == "SeasonalNaive":
        return np.array([vals[-12 + (i % 12)] for i in range(steps)]), idx
    if name == "ARIMA":
        try:
            p = ARIMA(vals, order=(1, 1, 1), seasonal_order=(1, 0, 0, 12)).fit().forecast(steps)
        except Exception:
            p = ARIMA(vals, order=(1, 1, 1)).fit().forecast(steps)
        return np.asarray(p), idx
    if name == "Ensemble":
        parts = [fit_forecast(n, s, steps)[0] for n in ["ARIMA", "ElasticNet", "RandomForest"]]
        return np.mean(parts, axis=0), idx
    # ML models with lag features, forecasting recursively
    X = [feats(list(vals[:t]), s.index[t].month) for t in range(12, len(vals))]
    model = make_price_ml(name).fit(np.array(X), vals[12:])
    hist, preds = list(vals), []
    for d in idx:
        p = float(model.predict(np.array([feats(hist, d.month)]))[0])
        preds.append(p); hist.append(p)
    return np.array(preds), idx

PRICE_MODELS = ["SeasonalNaive", "ARIMA", "ElasticNet", "RandomForest", "Ensemble"]
rows = []
for crop, s in series.items():
    tr, te = s[:-12], s[-12:]
    for name in PRICE_MODELS:
        p, _ = fit_forecast(name, tr, 12)
        rows.append({"crop": crop, "model": name, **metrics(te.values, p)})
price_results = pd.DataFrame(rows)
display(price_results.round(2))

best_price = price_results.loc[price_results.groupby("crop").RMSE.idxmin()].set_index("crop")["model"]
print("\nSelected price model per crop:"); print(best_price)

price_forecasts = {}
for crop, s in series.items():
    p, idx = fit_forecast(best_price[crop], s, 12)
    price_forecasts[crop] = pd.DataFrame({"date": idx, "forecast_price": p})

In [ ]:
def price_advisory(crop, top_n=3):
    f = price_forecasts[crop].copy()
    avg = f.forecast_price.mean()
    f["vs_avg_%"] = (f.forecast_price / avg - 1) * 100
    best = f.sort_values("forecast_price", ascending=False).head(top_n)
    msg = f"Best months to sell {crop}: " + ", ".join(
        f"{d:%B %Y} (P{p:.2f}/kg, {v:+.1f}% vs average)"
        for d, p, v in zip(best.date, best.forecast_price, best["vs_avg_%"]))
    return msg, f

for c in price_forecasts: print(price_advisory(c)[0])

crop_plot = list(series)[0]
ax = series[crop_plot][-36:].plot(label="history", figsize=(9, 3.5))
price_forecasts[crop_plot].set_index("date").forecast_price.plot(ax=ax, label="forecast", style="--")
ax.set_title(f"{crop_plot}: price per kg"); ax.legend(); plt.show()

## 6. Module 3: Weather-based planting advisory (agroclimatic matching)
Each month is scored by comparing Pagsanjan's average monthly temperature and rainfall with the crop's requirements.
> The ranges below are **placeholders**. Replace them with figures confirmed by the MAO / DA crop guides. For real-time use, replace `clim` with the weather-API forecast.

In [ ]:
# (temp_min, temp_max, monthly_rain_min_mm, monthly_rain_max_mm)  <-- verify with MAO / DA
CROP_REQ = {
    "Rice":     (22, 32, 150, 300),
    "Tomato":   (18, 30,  80, 200),
    "Eggplant": (22, 32,  80, 200),
    "Banana":   (24, 32, 100, 250),
    "Okra":     (24, 35,  80, 200),
}

def range_score(v, lo, hi):
    if lo <= v <= hi: return 1.0
    dist = lo - v if v < lo else v - hi
    return max(0.0, 1 - dist / (0.5 * (hi - lo)))

def planting_advisory(crop, climatology=clim, top_n=3):
    tmin, tmax, rmin, rmax = CROP_REQ[crop]
    sc = pd.DataFrame({"month": climatology.index,
                       "score": [(range_score(climatology.loc[m, "temp_c"], tmin, tmax) +
                                  range_score(climatology.loc[m, "rainfall_mm"], rmin, rmax)) / 2
                                 for m in climatology.index]})
    sc["month_name"] = sc.month.map(lambda m: calendar.month_name[m])
    best = sc.sort_values("score", ascending=False).head(top_n)
    return "Most favorable months to plant " + crop + ": " + ", ".join(
        f"{n} ({s*100:.0f}% match)" for n, s in zip(best.month_name, best.score)), sc

for c in CROP_REQ: print(planting_advisory(c)[0])

## 7. Combined estimate: harvest and income for one farmer
This is the function the app calls. It takes **land area** as the main input.

In [ ]:
def predict_harvest_income(crop, area_ha, harvest_month, barangay=None, rainfall_mm=None, temp_c=None, price_per_kg=None):
    barangay = barangay or df.barangay.mode()[0]
    r = rainfall_mm if rainfall_mm is not None else clim.loc[harvest_month, "rainfall_mm"]
    t = temp_c if temp_c is not None else clim.loc[harvest_month, "temp_c"]
    X = pd.DataFrame([{"crop": crop, "barangay": barangay,
                       "month_sin": np.sin(2*np.pi*harvest_month/12), "month_cos": np.cos(2*np.pi*harvest_month/12),
                       "rainfall_mm": r, "temp_c": t}])
    y_ha = float(harvest_model.predict(X[FEATS])[0])
    kg = max(y_ha, 0) * area_ha
    kg_low, kg_high = max(kg - harvest_rmse * area_ha, 0), kg + harvest_rmse * area_ha
    if price_per_kg is None:
        f = price_forecasts[crop]; hit = f[f.date.dt.month == harvest_month]
        price_per_kg = float(hit.forecast_price.iloc[0]) if len(hit) else float(series[crop][-12:].mean())
    return {"crop": crop, "area_ha": area_ha, "est_harvest_kg": round(kg, 1),
            "range_kg": (round(kg_low, 1), round(kg_high, 1)),
            "price_per_kg": round(price_per_kg, 2), "est_income_php": round(kg * price_per_kg, 2),
            "note": "Estimate only; not a guarantee of actual harvest or income."}

predict_harvest_income("Rice", area_ha=1.5, harvest_month=10)

## 8. Save results to Supabase
The app does **not** need to run Python. This notebook precomputes:
- `yield_estimates`: predicted **kg per hectare** for every crop × barangay × month (plus a municipality-wide `ALL` row),
- `price_forecasts`: next 12 months of price per kg per crop,
- `planting_advisory`: suitability score per crop × month,
- `model_metrics`: MAE / RMSE / MAPE of every model compared.

The app then computes **harvest = yield_kg_ha × area_ha** and **income = harvest × forecast_price** from the farmer's land area. Re-run this notebook whenever the MAO adds new records.

**Steps:** (1) run the SQL below once in Supabase → SQL Editor (it is also saved as `export/schema.sql`). (2) In Colab, click the 🔑 *Secrets* icon and add `SUPABASE_URL` and `SUPABASE_SERVICE_KEY` (Project Settings → API). (3) Set `PUSH_TO_SUPABASE = True` (the connection from section 2 is reused).
> The **service_role** key bypasses security rules. Keep it only in Colab Secrets. **Never put it in the Flutter app**, which uses the anon key and signed-in users only.

In [ ]:
PUSH_TO_SUPABASE = False

SCHEMA_SQL = """
create table if not exists price_forecasts (
  crop text not null, month date not null, forecast_price numeric not null,
  model text, generated_at timestamptz default now(), primary key (crop, month));
create table if not exists yield_estimates (
  crop text not null, barangay text not null, month int not null check (month between 1 and 12),
  yield_kg_ha numeric not null, rmse_kg_ha numeric, model text,
  generated_at timestamptz default now(), primary key (crop, barangay, month));
create table if not exists planting_advisory (
  crop text not null, month int not null check (month between 1 and 12), score numeric not null,
  generated_at timestamptz default now(), primary key (crop, month));
create table if not exists model_metrics (
  module text not null, model text not null, crop text not null default 'ALL',
  mae numeric, rmse numeric, mape_pct numeric, selected boolean default false,
  generated_at timestamptz default now(), primary key (module, model, crop));

alter table price_forecasts  enable row level security;
alter table yield_estimates  enable row level security;
alter table planting_advisory enable row level security;
alter table model_metrics    enable row level security;
create policy "read price_forecasts"  on price_forecasts  for select to authenticated using (true);
create policy "read yield_estimates"  on yield_estimates  for select to authenticated using (true);
create policy "read planting_advisory" on planting_advisory for select to authenticated using (true);
create policy "read model_metrics"    on model_metrics    for select to authenticated using (true);
"""
os.makedirs("export", exist_ok=True)
open("export/schema.sql", "w").write(SCHEMA_SQL)

# ---- yield per hectare: crop x barangay x month (typical weather of that month) ----
pairs = df[["crop", "barangay"]].drop_duplicates()
grid = pairs.merge(pd.DataFrame({"month": range(1, 13)}), how="cross")
grid["rainfall_mm"] = grid.month.map(clim.rainfall_mm)
grid["temp_c"] = grid.month.map(clim.temp_c)
grid["month_sin"] = np.sin(2*np.pi*grid.month/12)
grid["month_cos"] = np.cos(2*np.pi*grid.month/12)
grid["yield_kg_ha"] = np.clip(harvest_model.predict(grid[FEATS]), 0, None)
muni = grid.groupby(["crop", "month"], as_index=False).yield_kg_ha.mean()
muni["barangay"] = "ALL"
yield_estimates = pd.concat([grid[["crop", "barangay", "month", "yield_kg_ha"]], muni], ignore_index=True)
yield_estimates["yield_kg_ha"] = yield_estimates.yield_kg_ha.round(1)
yield_estimates["rmse_kg_ha"] = round(float(harvest_rmse), 1)
yield_estimates["model"] = best_harvest

# ---- price forecasts ----
price_rows = [{"crop": c, "month": d.strftime("%Y-%m-01"), "forecast_price": round(float(p), 2), "model": best_price[c]}
              for c, f in price_forecasts.items() for d, p in zip(f.date, f.forecast_price)]

# ---- planting advisory ----
plant_rows = [{"crop": c, "month": int(m), "score": round(float(sc), 3)}
              for c in CROP_REQ for m, sc in zip(*[planting_advisory(c)[1][k] for k in ("month", "score")])]

# ---- model metrics ----
metric_rows = [{"module": "harvest", "model": m, "crop": "ALL", "mae": round(r["MAE"], 3), "rmse": round(r["RMSE"], 3),
                "mape_pct": round(r["MAPE_%"], 3), "selected": m == best_harvest} for m, r in harvest_results.iterrows()]
metric_rows += [{"module": "price", "model": r.model, "crop": r.crop, "mae": round(r.MAE, 3), "rmse": round(r.RMSE, 3),
                 "mape_pct": round(r["MAPE_%"], 3), "selected": best_price[r.crop] == r.model} for _, r in price_results.iterrows()]

yield_rows = yield_estimates.to_dict(orient="records")
yield_estimates.to_csv("export/yield_estimates.csv", index=False)
print({"yield_estimates": len(yield_rows), "price_forecasts": len(price_rows),
       "planting_advisory": len(plant_rows), "model_metrics": len(metric_rows)})

if PUSH_TO_SUPABASE:
    sb = get_sb()
    send_rows(sb, "yield_estimates", yield_rows, conflict="crop,barangay,month")
    send_rows(sb, "price_forecasts", price_rows, conflict="crop,month")
    send_rows(sb, "planting_advisory", plant_rows, conflict="crop,month")
    send_rows(sb, "model_metrics", metric_rows, conflict="module,model,crop")

    # GET it back to confirm the data really arrived
    for t in ["yield_estimates", "price_forecasts", "planting_advisory", "model_metrics"]:
        n = sb.table(t).select("*", count="exact").limit(1).execute().count
        print(f"{t}: {n} rows now in Supabase")
    display(pd.DataFrame(sb.table("model_metrics").select("*").eq("selected", True).execute().data))

**How the Flutter app uses it** (`supabase_flutter`, signed-in user):
```dart
final y = await supabase.from('yield_estimates').select()
    .eq('crop', crop).eq('barangay', barangay).eq('month', month).maybeSingle();
final p = await supabase.from('price_forecasts').select()
    .eq('crop', crop).eq('month', '2026-12-01').maybeSingle();   // first day of the month
final harvestKg = y['yield_kg_ha'] * areaHa;
final income    = harvestKg * p['forecast_price'];
```
If a barangay has no row, query `barangay = 'ALL'`.

## 9. Export files (models, JSON, SQL)

In [ ]:
os.makedirs("export", exist_ok=True)
joblib.dump(harvest_model, "export/harvest_model.pkl")
joblib.dump({"features": FEATS, "climatology": clim, "rmse": harvest_rmse, "barangay_default": df.barangay.mode()[0]},
            "export/harvest_meta.pkl")

out = {
    "price_forecasts": {c: [{"month": d.strftime("%Y-%m"), "price": round(float(p), 2)}
                            for d, p in zip(f.date, f.forecast_price)] for c, f in price_forecasts.items()},
    "price_advisory": {c: price_advisory(c)[0] for c in price_forecasts},
    "planting_advisory": {c: planting_advisory(c)[0] for c in CROP_REQ},
    "harvest_model_metrics": harvest_results.round(3).to_dict(orient="index"),
    "price_model_metrics": price_results.round(3).to_dict(orient="records"),
    "selected_models": {"harvest": best_harvest, "price": best_price.to_dict()},
}
json.dump(out, open("export/advisories.json", "w"), indent=2, default=str)
harvest_results.to_csv("export/harvest_model_comparison.csv")
price_results.to_csv("export/price_model_comparison.csv", index=False)

shutil.make_archive("dss_models", "zip", "export")
try:
    from google.colab import files
    files.download("dss_models.zip")
except Exception:
    print("Saved dss_models.zip")

## 10. Reuse later (load the saved model without retraining)
```python
import joblib, pandas as pd
model = joblib.load("harvest_model.pkl")
X = pd.DataFrame([{"crop": "Rice", "barangay": "Barangay 1", "month_sin": 0.0, "month_cos": 1.0,
                   "rainfall_mm": 200, "temp_c": 27}])
print(model.predict(X)[0] * 1.5)   # kg for 1.5 ha
```
The new data must keep the same columns and categories as the training data. To learn from new MAO records, add them to the CSVs and re-run the notebook.